# Hyperparameter search with Optuna + the scikit-learn contract

`sklm` estimators honour the scikit-learn parameter protocol, and the config objects
(`TrainingConfig`, `LoRAConfig`, `GenerationConfig`) subclass `BaseEstimator`. So **every nested
field is addressable** through the usual `__` path, and any sklearn-compatible search can tune it.
Inside a `Pipeline` step named `lm`:

- `lm__precision` — a flat field on the estimator.
- `lm__training__epochs` — the `epochs` field of the nested `TrainingConfig`.
- `lm__lora__rank` — the `rank` field of the nested `LoRAConfig`.

You declare the fixed hyperparameters once on the estimator and put only the swept fields in
`param_distributions`; there is no need to re-instantiate a whole config per trial.

## How `OptunaSearchCV` searches

For each trial $t$, Optuna proposes a configuration $\theta_t$ and evaluates it by $k$-fold
cross-validation, scoring the mean validation accuracy

$$ s(\theta_t) = \frac{1}{k}\sum_{i=1}^{k} \operatorname{accuracy}\!\left(\text{fold } i;\, \theta_t\right). $$

The default sampler here is **TPE** (Tree-structured Parzen Estimator). Instead of grid or random
search, TPE models the past trials with two densities over the hyperparameters — $l(\theta)$ fit to
the better-scoring trials and $g(\theta)$ to the rest — and proposes the $\theta$ that maximizes the
ratio $l(\theta)\,/\,g(\theta)$, concentrating samples where good scores have already appeared.

With $k$-fold CV inside each of $T$ trials, the search fine-tunes the model $T \times k$ times — the
dominant cost. distilgpt2 on Iris is a weak classifier; the point here is the **tuning ergonomics**,
not the accuracy.

> This notebook uses `gpt2-large` and `15 x 4 = 60` fine-tunes, which is slow. For a quick local
> run, switch `model` to `gabfssilva/distilgpt2` and lower `n_trials` / `cv`.

In [ ]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from optuna import create_study
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
from optuna.integration import OptunaSearchCV
from optuna.samplers import TPESampler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklm import (
    KeyValueSerializer,
    LanguageModelClassifier,
    LoRAConfig,
    SpacedDigits,
    TqdmCallback,
    TrainingConfig,
)

SEED = 42

## Estimator, pipeline, and search space

`SpacedDigits` serializes each number one token per digit (`5 . 1`), which can help a small model
read magnitudes. The fixed knobs live on the estimator; the swept fields — learning rate, schedule,
epochs, batch size, and two LoRA fields — are addressed by their `__` paths.

In [ ]:
clf = LanguageModelClassifier(
    model="gabfssilva/distilgpt2",
    backend="mlx",
    precision="fp32",
    serializer=KeyValueSerializer(number=SpacedDigits()),
    training=TrainingConfig(augmentation_factor=12),
    lora=LoRAConfig(),
    callback=TqdmCallback(n_train_examples=3),
    random_state=SEED,
)

pipe = Pipeline([("lm", clf)])

param_distributions = {
    "lm__training__learning_rate": CategoricalDistribution([0.0001, 0.0002, 0.0003]),
    "lm__training__lr_scheduler": CategoricalDistribution(["linear", "cosine"]),
    "lm__training__epochs": IntDistribution(2, 5),
    "lm__training__batch_size": CategoricalDistribution([8, 16]),
    "lm__lora__rank": CategoricalDistribution([16, 32]),
    "lm__lora__dropout": FloatDistribution(0.05, 0.2),
}

## Run the search

`OptunaSearchCV` clones the pipeline for every (trial, fold) pair, so each fit starts from the base
model with no state leaking between trials.

In [ ]:
search = OptunaSearchCV(
    pipe,
    param_distributions,
    cv=4,
    scoring="accuracy",
    n_trials=15,
    random_state=SEED,
    verbose=2,
    study=create_study(
        direction="maximize",
        sampler=TPESampler(multivariate=True, seed=SEED),
    ),
)

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target_names[iris.target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

search.fit(X_train, y_train)

## Best configuration

The held-out test accuracy uses the best configuration refit on the full training split.

In [ ]:
print(f"best params:      {search.best_params_}")
print(f"best CV accuracy: {search.best_score_:.3f}")
print(f"test accuracy:    {accuracy_score(y_test, search.predict(X_test)):.3f}")